# Town demand

The town center consumes 1 of every non-fertilizer product every 24 turns (once per day) for the whole 30-day season. Every 3 days a new shop unlocks, drawn uniformly with replacement from the 8 shop types, capped at 8 total instances. Each shop consumes 1 of every product it demands every 4 turns (6/day per demanded product); single-product shops consume 2x.

Because shops are drawn with replacement, the per-resource demand distribution has large variance across seasons. Simulate 5000 seasons to estimate mean and 5-95% percentile bands per resource.

In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from kaggriculture.env.constants import (
    MAX_SHOP_INSTANCES,
    SHOPS,
    TOWN_CENTER_PRODUCTS,
)

FIG_DIR = Path.cwd().parent / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
SEASON_DAYS = 30
TURNS_PER_DAY = 24
TOTAL_TURNS = SEASON_DAYS * TURNS_PER_DAY
SHOP_UNLOCK_INTERVAL = 3  # days
SHOP_CONSUME_INTERVAL = 4  # turns
TOWN_CENTER_INTERVAL = 24  # turns
SHOP_NAMES = sorted(SHOPS)
PRODUCTS = [*TOWN_CENTER_PRODUCTS, "FERTILIZER"]


def simulate_season_demand(seed: int) -> dict[str, int]:
    rng = random.Random(seed)
    demand = dict.fromkeys(PRODUCTS, 0)
    unlocked: list[str] = []
    for step in range(TOTAL_TURNS):
        day = step // TURNS_PER_DAY
        # Town shops consume at their interval
        if step % SHOP_CONSUME_INTERVAL == 0:
            for shop_name in unlocked:
                products = SHOPS[shop_name]
                mult = 2 if len(products) == 1 else 1
                for item in products:
                    demand[item] += mult
        # Town center consumes 1 of every non-fertilizer product
        if step % TOWN_CENTER_INTERVAL == 0:
            for item in TOWN_CENTER_PRODUCTS:
                demand[item] += 1
        # End-of-day: maybe unlock a new shop
        if (step + 1) % TURNS_PER_DAY == 0:
            next_day = day + 1
            if (
                next_day > 0
                and next_day % SHOP_UNLOCK_INTERVAL == 0
                and len(unlocked) < MAX_SHOP_INSTANCES
            ):
                unlocked.append(rng.choice(SHOP_NAMES))
    return demand


n_seasons = 5000
rows = [simulate_season_demand(s) for s in range(n_seasons)]
demand_df = pd.DataFrame(rows)
demand_df.describe(percentiles=[0.05, 0.5, 0.95]).round(1)

In [ ]:
products_order = [p for p in PRODUCTS if p != "FERTILIZER"]
means = demand_df[products_order].mean()
p05 = demand_df[products_order].quantile(0.05)
p95 = demand_df[products_order].quantile(0.95)

fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(products_order))
ax.bar(x, means, color="#1f77b4", label="mean")
ax.errorbar(
    x, means, yerr=[means - p05, p95 - means], fmt="none", color="black", capsize=4, label="5-95%"
)
ax.set_xticks(x)
ax.set_xticklabels(products_order, rotation=30, ha="right")
ax.set_ylabel("season total units consumed by town")
ax.set_title(f"Per-resource town demand over {n_seasons} simulated seasons")
ax.grid(True, axis="y", alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "town-demand.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# How often does a season have NO shop demanding a given resource beyond the town-center baseline?
town_center_baseline = SEASON_DAYS  # 1 per day * 30 days
for product in products_order:
    if product in TOWN_CENTER_PRODUCTS:
        shop_only_demand = demand_df[product] - town_center_baseline
    else:
        shop_only_demand = demand_df[product]
    zero_share = (shop_only_demand == 0).mean() * 100
    print(f"{product:12s}: {zero_share:5.1f}% of seasons have zero shop demand")

## Takeaways

- With replacement draws and only 8 unlock slots, the town shop composition has large variance. Wool is only demanded by the yarn store (single-product, 2x consumption); a nonzero share of seasons never unlock any yarn store, meaning wool demand is only the town center's 1/day.
- Wheat and strawberry are consumed by the most shop types (wheat: 5 of 8 shop templates; strawberry: 4). They are the most reliably-demanded resources.
- The town center provides a floor of 24 units per non-fertilizer product per season (once per day). That is the guaranteed sink for any production.
- A production plan that assumes uniform demand will over-supply low-demand resources in the low-tail seasons (yarn store never appears). Adapting mid-season to the observed unlock pattern is worth measurable Elo.